## **Character distribution — original transcription vs. synthetic-seed lines vs. COMETA corpus**

Compare the per-character frequency across three sources to make sure the lines we feed the synthetic-image generator are linguistically representative of (a) the real manuscript we're targeting and (b) the broader medieval corpus:

1. **AlbucE.txt** — the human transcription of the reference manuscript (single file).
2. **`cometa_categorized.json` seed lines** — the filtered/curated subset we draw synthetic text from (output of `corpus_categorization`).
3. **COMETA medieval corpus** — every `*.txt` file under `data/raw/COMETA_medieval_corpus/` (the unfiltered superset).

Sections:

1. **Setup** — imports + paths.
2. **Load each source** — extract raw text from disk / JSON.
3. **Per-source character frequency** — counts and fractions.
4. **Side-by-side comparison** — merged table, set differences.
5. **Visual comparison** — bar charts on linear and log scales.
6. **Drill-down: characters unique to a source** — what one source has that the others don't.

### 1. Setup

In [ ]:
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"])

# Make `src.*` importable when the kernel was launched outside the project root.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ALBUC_TXT       = PROJECT_ROOT / "data/raw/AlbucE.txt"
CATEGORIZED_JSON = PROJECT_ROOT / "data/processed/synthetic_seeds/categorize_20260613_214958/cometa_categorized.json"
COMETA_DIR      = PROJECT_ROOT / "data/raw/COMETA_medieval_corpus"

print(f"AlbucE:           {ALBUC_TXT}")
print(f"Categorized seed: {CATEGORIZED_JSON}")
print(f"COMETA corpus:    {COMETA_DIR}")

### 2. Load each source

Each loader returns the raw text concatenation that we'll count over. Loading the COMETA corpus walks every `*.txt` file (the matching `.pdf` files are ignored).

In [ ]:
def load_albuc(path: Path) -> str:
    """Single-file plain-text transcription of the target manuscript."""
    return path.read_text(encoding="utf-8")


def load_categorized_seeds(path: Path) -> str:
    """Concatenated text from every record in the categorized seed JSON."""
    doc = json.loads(path.read_text(encoding="utf-8"))
    samples = doc.get("samples") or {}
    return "\n".join(s.get("text", "") for s in samples.values())


def load_cometa_corpus(dirpath: Path) -> str:
    """Concatenated text from every *.txt file under the COMETA corpus dir."""
    parts = []
    for f in sorted(dirpath.glob("*.txt")):
        parts.append(f.read_text(encoding="utf-8", errors="replace"))
    return "\n".join(parts)


sources = {
    "AlbucE":           load_albuc(ALBUC_TXT),
    "categorized_seed": load_categorized_seeds(CATEGORIZED_JSON),
    "COMETA_full":      load_cometa_corpus(COMETA_DIR),
}

print(f"{'source':<20} {'chars':>12} {'unique':>8} {'lines':>10}")
for name, text in sources.items():
    n_lines = text.count("\n") + 1
    print(f"{name:<20} {len(text):>12,} {len(set(text)):>8} {n_lines:>10,}")

### 3. Per-source character frequency

For each source, count every character (yes, including whitespace and punctuation) and convert to a fraction of the total so the three sources are comparable despite their different sizes.

In [ ]:
def char_distribution(text: str) -> pd.DataFrame:
    """Return a DataFrame with columns ['char', 'count', 'fraction']."""
    c = Counter(text)
    total = sum(c.values())
    df = pd.DataFrame(
        [(ch, n, n / total) for ch, n in c.items()],
        columns=["char", "count", "fraction"],
    )
    return df.sort_values("count", ascending=False).reset_index(drop=True)


per_source = {name: char_distribution(text) for name, text in sources.items()}

# Top-N per source side by side — using the UNION of each source's top N
# and then looking each char up in the FULL distribution. Otherwise a
# char that's 11th in one source and 8th in another gets a fabricated 0
# from `df.head(N)` truncation (which would falsely tell you AlbucE has
# no `n`).
top_n = 10
top_chars = list({c for df in per_source.values() for c in df["char"].head(top_n)})
preview = pd.concat(
    {name: df.set_index("char")["fraction"].rename(name) for name, df in per_source.items()},
    axis=1,
).fillna(0.0).loc[top_chars]
preview = preview.sort_values("categorized_seed", ascending=False)
preview

### 4. Side-by-side comparison

One row per character, one column per source — fractions so the magnitudes are comparable. Characters absent from a source get 0. Sorted by the categorized-seed fraction (descending) since that's the distribution that actually drives synthetic generation.

In [ ]:
merged = pd.concat(
    {name: df.set_index("char")["fraction"].rename(name) for name, df in per_source.items()},
    axis=1,
).fillna(0.0)

# Add a 'rendering' column so non-printable / whitespace chars are obvious in the table.
def _render(ch):
    if ch == " ":
        return "<SPACE>"
    if ch == "\n":
        return "<NEWLINE>"
    if ch == "\t":
        return "<TAB>"
    if ch == "\r":
        return "<CR>"
    if ord(ch) < 32 or ord(ch) == 127:
        return f"<U+{ord(ch):04X}>"
    return ch

merged.insert(0, "render", [_render(ch) for ch in merged.index])
merged = merged.sort_values("categorized_seed", ascending=False)
merged.head(30)

In [ ]:
# Same merged table but in absolute counts (handy when comparing 'how many
# Roman numerals are actually in each source' rather than relative fractions).
merged_counts = pd.concat(
    {name: df.set_index("char")["count"].rename(name) for name, df in per_source.items()},
    axis=1,
).fillna(0).astype(int)
merged_counts.insert(0, "render", [_render(ch) for ch in merged_counts.index])
merged_counts = merged_counts.loc[merged.index]   # keep the same row order
merged_counts.head(30)

#### Export both tables to CSV

Saves the full fraction table + the counts table side by side to `tests/character_distribution/`. One file per run timestamp so you can keep multiple snapshots over time.

In [ ]:
import datetime

out_dir = PROJECT_ROOT / "tests" / "character_distribution"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Combine fractions and counts into a single wide CSV so a reader can
# see absolute volume and relative share for each (char, source) cell.
frac_cols = {f"frac_{c}": merged[c] for c in merged.columns if c != "render"}
count_cols = {f"count_{c}": merged_counts[c] for c in merged_counts.columns if c != "render"}
wide = pd.DataFrame({"render": merged["render"], **frac_cols, **count_cols})
wide.index.name = "char"
wide["codepoint"] = [f"U+{ord(c):04X}" for c in wide.index]
# Move codepoint to position 1 (right after render)
wide = wide[["render", "codepoint"] + [c for c in wide.columns if c not in ("render", "codepoint")]]

out_csv = out_dir / f"character_distribution_{stamp}.csv"
wide.to_csv(out_csv, encoding="utf-8")
print(f"wrote {out_csv}")
print(f"rows: {len(wide)}  cols: {list(wide.columns)}")
wide.head(20)

### 5. Visual comparison

Two views of the same data:

- **Top-N bar chart** — the most frequent characters in any source, plotted side by side. Whitespace and lowercase letters dominate.
- **Log-scale full distribution** — every character on the y-axis, log-scaled fraction on the x-axis. Lets you eyeball the long tail (rare characters, capital letters, punctuation) without the bulk drowning them out.

In [ ]:
import numpy as np

# Pick the union of the top-N characters across all sources so the chart
# shows the same rows in every panel.
top_n = 20
top_chars = list({c for df in per_source.values() for c in df["char"].head(top_n)})
top_chars = sorted(top_chars, key=lambda c: -merged.loc[c, "categorized_seed"])
chart = merged.loc[top_chars]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(chart))
bar_w = 0.27
source_names = [c for c in chart.columns if c != "render"]
colors = ["#3a6a9a", "#6a8a5a", "#9a6a4a"]
for i, src in enumerate(source_names):
    ax.bar(x + (i - 1) * bar_w, chart[src], width=bar_w, label=src, color=colors[i])
ax.set_xticks(x)
ax.set_xticklabels(chart["render"], rotation=45, ha="right")
ax.set_ylabel("fraction of all characters")
ax.set_title(f"Top-{top_n} character fractions per source")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Log-scale view of the WHOLE distribution. Use a tiny floor for absent
# chars so log doesn't blow up.
floor = 1e-7
sorted_idx = merged["categorized_seed"].sort_values(ascending=False).index
data = merged.loc[sorted_idx, source_names].clip(lower=floor)

fig, ax = plt.subplots(figsize=(13, max(6, 0.18 * len(sorted_idx))))
y = np.arange(len(sorted_idx))
for i, src in enumerate(source_names):
    ax.scatter(data[src], y, label=src, alpha=0.7, s=25, color=colors[i])
ax.set_xscale("log")
ax.set_yticks(y)
ax.set_yticklabels([_render(ch) for ch in sorted_idx], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("fraction (log scale)")
ax.set_title("Per-character fraction, log scale, all characters")
ax.legend()
ax.grid(True, axis="x", which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### 6. Drill-down: characters unique to a source

Set-arithmetic on the character vocabularies. Helpful for spotting weird OCR/transcription artefacts that exist in only one source — e.g. a stray non-medieval punctuation mark that snuck into AlbucE but isn't anywhere in the COMETA corpus, or an exotic character that the synthetic generator should learn to handle but the seed JSON has filtered out.

In [ ]:
char_sets = {name: set(df["char"]) for name, df in per_source.items()}

# Each row: (set name, |set|, set rendered)
def _show(label, s):
    print(f"{label:<40}  n={len(s):>3}")
    rendered = sorted(s, key=lambda c: -merged.get(c, pd.Series([0])).max() if c in merged.index else 0)
    for ch in rendered:
        n_each = "  ".join(f"{src}={int(merged_counts.at[ch, src]) if ch in merged_counts.index else 0:>5}" for src in source_names)
        print(f"  {_render(ch):<10}  {n_each}")

_show("only in AlbucE",
      char_sets["AlbucE"] - char_sets["categorized_seed"] - char_sets["COMETA_full"])
print()
_show("only in categorized_seed",
      char_sets["categorized_seed"] - char_sets["AlbucE"] - char_sets["COMETA_full"])
print()
_show("only in COMETA_full",
      char_sets["COMETA_full"] - char_sets["AlbucE"] - char_sets["categorized_seed"])
print()
_show("in COMETA_full but missing from categorized_seed",
      char_sets["COMETA_full"] - char_sets["categorized_seed"])
print()
_show("in COMETA_full but missing from AlbucE",
      char_sets["COMETA_full"] - char_sets["AlbucE"])

### What to look at

- If `categorized_seed` is missing characters that exist in `AlbucE`, the synthetic generator will never produce them — but the OCR target needs to recognise them. Either add to the seed JSON or accept the gap.
- If `categorized_seed` has characters that `AlbucE` doesn't, you're training the OCR on inputs the deployment target will never see — usually fine, but bias to keep an eye on.
- Big fraction gaps on a common character (e.g. `s` differs by more than ~2 percentage points across the three sources) usually means the categorization filter biased the line selection more than expected.